In [1]:
!pip install -q -U transformers datasets accelerate evaluate wandb

# IMPORTANT (first time only): Colab preloads an older transformers version.
# After this cell finishes, go to Runtime > Restart session, then re-run
# all cells from the top (skip re-running this install cell if you like —
# it's already done). This ensures the upgraded version is actually loaded.
import transformers
print("transformers version:", transformers.__version__)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 43.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.4/604.4 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.1/26.1 MB 31.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 18.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
cudf-cu13 26.6.0 requires pyarrow<24,>=19.0.0, but you have pyarrow 25.0.1 which is incompatible.
google-adk 2.7

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os
import json
import collections
import subprocess

import numpy as np
import torch
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForQuestionAnswering,
    TrainingArguments,
    Trainer,
)

MODEL_CHECKPOINT = "bert-base-uncased"   # swap to "roberta-base" or
                                          # "microsoft/deberta-v3-base" later

PROJECT_DIR = "/content/drive/MyDrive/CS4248 Group Project"
DATA_DIR = os.path.join(PROJECT_DIR, "qa")
TRAIN_FILE = os.path.join(DATA_DIR, "train-v1.1.json")
DEV_FILE = os.path.join(DATA_DIR, "dev-v1.1.json")
EVAL_SCRIPT = os.path.join(DATA_DIR, "evaluate-v2.0.py")

MAX_LENGTH = 384
DOC_STRIDE = 128
BATCH_SIZE = 8                  # lowered from 16 to reduce activation memory
GRADIENT_ACCUMULATION_STEPS = 2  # keeps effective batch size at 8*2=16
LEARNING_RATE = 3e-5
NUM_EPOCHS = 2
N_BEST = 20
MAX_ANSWER_LENGTH = 30
OUTPUT_DIR = os.path.join(PROJECT_DIR, "qa_model")  # saved straight to Drive, survives session end
PREDICTIONS_FILE = "predictions.json"
USE_WANDB = False   # set True if you run wandb.login() in the next cell

# Set these to small ints (e.g. 2000 / 500) to smoke-test the pipeline fast
# before committing to a full run on the full ~87k/10k SQuAD examples.
MAX_TRAIN_SAMPLES = None
MAX_EVAL_SAMPLES = None

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
assert torch.cuda.is_available(), (
    "No GPU detected. Go to Runtime > Change runtime type > Hardware "
    "accelerator > GPU (T4), save, reconnect, then re-run from the top. "
    "Training BERT on CPU will exhaust Colab's RAM and crash the session."
)
print("GPU:", torch.cuda.get_device_name(0))

GPU: Tesla T4


In [5]:
# Optional: experiment tracking
# import wandb
# wandb.login()
# USE_WANDB = True

In [6]:
def load_squad_json(path):
    with open(path) as f:
        squad = json.load(f)
    ids, questions, contexts, answers = [], [], [], []
    for article in squad["data"]:
        for para in article["paragraphs"]:
            context = para["context"]
            for qa in para["qas"]:
                ids.append(qa["id"])
                questions.append(qa["question"])
                contexts.append(context)
                answers.append(
                    {
                        "text": [a["text"] for a in qa["answers"]],
                        "answer_start": [a["answer_start"] for a in qa["answers"]],
                    }
                )
    return Dataset.from_dict(
        {"id": ids, "question": questions, "context": contexts, "answers": answers}
    )


print("Loading data...")
raw_datasets = DatasetDict(
    {
        "train": load_squad_json(TRAIN_FILE),
        "validation": load_squad_json(DEV_FILE),
    }
)

if MAX_TRAIN_SAMPLES is not None:
    raw_datasets["train"] = raw_datasets["train"].select(range(MAX_TRAIN_SAMPLES))
if MAX_EVAL_SAMPLES is not None:
    raw_datasets["validation"] = raw_datasets["validation"].select(range(MAX_EVAL_SAMPLES))

raw_datasets

Loading data...


DatasetDict({
    train: Dataset({
        features: ['id', 'question', 'context', 'answers'],
        num_rows: 87599
    })
    validation: Dataset({
        features: ['id', 'question', 'context', 'answers'],
        num_rows: 10570
    })
})

In [7]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)
assert tokenizer.is_fast, "Need a fast tokenizer for offset mapping"

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [8]:
def prepare_train_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    offset_mapping = tokenized.pop("offset_mapping")

    start_positions, end_positions = [], []
    for i, offsets in enumerate(offset_mapping):
        sample_idx = sample_map[i]
        answer = examples["answers"][sample_idx]
        start_char = answer["answer_start"][0]
        end_char = start_char + len(answer["text"][0])
        sequence_ids = tokenized.sequence_ids(i)

        idx = 0
        while sequence_ids[idx] != 1:
            idx += 1
        context_start = idx
        while idx < len(sequence_ids) and sequence_ids[idx] == 1:
            idx += 1
        context_end = idx - 1

        if offsets[context_start][0] > start_char or offsets[context_end][1] < end_char:
            start_positions.append(0)
            end_positions.append(0)
        else:
            idx = context_start
            while idx <= context_end and offsets[idx][0] <= start_char:
                idx += 1
            start_positions.append(idx - 1)
            idx = context_end
            while idx >= context_start and offsets[idx][1] >= end_char:
                idx -= 1
            end_positions.append(idx + 1)

    tokenized["start_positions"] = start_positions
    tokenized["end_positions"] = end_positions
    return tokenized


def prepare_validation_features(examples):
    questions = [q.lstrip() for q in examples["question"]]
    tokenized = tokenizer(
        questions,
        examples["context"],
        max_length=MAX_LENGTH,
        truncation="only_second",
        stride=DOC_STRIDE,
        return_overflowing_tokens=True,
        return_offsets_mapping=True,
        padding="max_length",
    )

    sample_map = tokenized.pop("overflow_to_sample_mapping")
    example_ids = []
    for i in range(len(tokenized["input_ids"])):
        sample_idx = sample_map[i]
        example_ids.append(examples["id"][sample_idx])
        sequence_ids = tokenized.sequence_ids(i)
        tokenized["offset_mapping"][i] = [
            o if sequence_ids[k] == 1 else None
            for k, o in enumerate(tokenized["offset_mapping"][i])
        ]

    tokenized["example_id"] = example_ids
    return tokenized


print("Tokenizing train set...")
train_dataset = raw_datasets["train"].map(
    prepare_train_features,
    batched=True,
    remove_columns=raw_datasets["train"].column_names,
)

print("Tokenizing validation set...")
validation_dataset = raw_datasets["validation"].map(
    prepare_validation_features,
    batched=True,
    remove_columns=raw_datasets["validation"].column_names,
)

Tokenizing train set...


Map:   0%|          | 0/87599 [00:00<?, ? examples/s]

Tokenizing validation set...


Map:   0%|          | 0/10570 [00:00<?, ? examples/s]

In [9]:
import dataclasses
import inspect

model = AutoModelForQuestionAnswering.from_pretrained(MODEL_CHECKPOINT)

# Build TrainingArguments kwargs defensively: different transformers versions
# rename/rearrange some fields (eval_strategy vs evaluation_strategy,
# warmup_ratio availability, etc). Only pass kwargs the installed version
# actually accepts, so this works regardless of which version ended up loaded.
ta_fields = {f.name for f in dataclasses.fields(TrainingArguments)}


def pick(*name_value_pairs):
    for name, value in name_value_pairs:
        if name in ta_fields:
            return {name: value}
    return {}


ta_kwargs = {"output_dir": OUTPUT_DIR}
ta_kwargs.update(pick(("eval_strategy", "no"), ("evaluation_strategy", "no")))
ta_kwargs.update(pick(("save_strategy", "epoch")))
ta_kwargs["learning_rate"] = LEARNING_RATE
ta_kwargs["per_device_train_batch_size"] = BATCH_SIZE
ta_kwargs["per_device_eval_batch_size"] = BATCH_SIZE
ta_kwargs["gradient_accumulation_steps"] = GRADIENT_ACCUMULATION_STEPS
ta_kwargs["num_train_epochs"] = NUM_EPOCHS
ta_kwargs["weight_decay"] = 0.01
ta_kwargs.update(pick(("warmup_ratio", 0.1)))
ta_kwargs.update(pick(("fp16", torch.cuda.is_available())))
ta_kwargs.update(pick(("gradient_checkpointing", True)))  # trades compute for a lot less activation memory
ta_kwargs.update(pick(("dataloader_num_workers", 0)))
ta_kwargs.update(pick(("report_to", "wandb" if USE_WANDB else "none")))
ta_kwargs.update(pick(("run_name", f"{MODEL_CHECKPOINT}-squad-v1.1")))

dropped = set(["eval_strategy", "warmup_ratio", "fp16", "gradient_checkpointing", "report_to", "run_name"]) - set(ta_kwargs)
if dropped:
    print(f"Note: this transformers version doesn't support {dropped}; skipping them.")

args = TrainingArguments(**ta_kwargs)

# Trainer renamed the `tokenizer` kwarg to `processing_class` in newer
# versions; detect which one the installed Trainer actually accepts.
trainer_params = set(inspect.signature(Trainer.__init__).parameters)
tokenizer_kwarg = "processing_class" if "processing_class" in trainer_params else "tokenizer"

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    **{tokenizer_kwarg: tokenizer},
)

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForQuestionAnswering LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
qa_outputs.weight                          | MISSING    | 
qa_outputs.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly i

Note: this transformers version doesn't support {'warmup_ratio'}; skipping them.


In [10]:
print("Training...")
trainer.train()
trainer.save_model(OUTPUT_DIR)
print(f"Model saved to {OUTPUT_DIR}")

Training...


Step,Training Loss
500,2.286103
1000,1.459692
1500,1.304430
2000,1.243590
2500,1.199929
3000,1.161986
3500,1.092897
4000,1.099779
4500,1.062876
5000,1.050281


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved to /content/drive/MyDrive/CS4248 Group Project/qa_model


In [11]:
print("Running inference on validation set...")
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

eval_set_for_model = validation_dataset.remove_columns(["example_id", "offset_mapping"])
eval_set_for_model.set_format("torch")

all_start_logits, all_end_logits = [], []
eval_batch_size = BATCH_SIZE
with torch.no_grad():
    for i in range(0, len(eval_set_for_model), eval_batch_size):
        batch = eval_set_for_model[i : i + eval_batch_size]
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        all_start_logits.append(outputs.start_logits.cpu().numpy())
        all_end_logits.append(outputs.end_logits.cpu().numpy())

start_logits = np.concatenate(all_start_logits, axis=0)
end_logits = np.concatenate(all_end_logits, axis=0)

Running inference on validation set...


In [12]:
def postprocess_qa_predictions(examples, features, all_start_logits, all_end_logits):
    example_id_to_index = {k: i for i, k in enumerate(examples["id"])}
    features_per_example = collections.defaultdict(list)
    for i, feature_id in enumerate(features["example_id"]):
        features_per_example[example_id_to_index[feature_id]].append(i)

    predictions = {}
    for example_index, example in enumerate(examples):
        context = example["context"]
        feature_indices = features_per_example[example_index]

        valid_answers = []
        for feature_index in feature_indices:
            start_logit = all_start_logits[feature_index]
            end_logit = all_end_logits[feature_index]
            offsets = features[feature_index]["offset_mapping"]

            start_indexes = np.argsort(start_logit)[-1 : -N_BEST - 1 : -1].tolist()
            end_indexes = np.argsort(end_logit)[-1 : -N_BEST - 1 : -1].tolist()
            for start_index in start_indexes:
                for end_index in end_indexes:
                    if (
                        start_index >= len(offsets)
                        or end_index >= len(offsets)
                        or offsets[start_index] is None
                        or offsets[end_index] is None
                    ):
                        continue
                    if end_index < start_index or end_index - start_index + 1 > MAX_ANSWER_LENGTH:
                        continue
                    start_char, end_char = offsets[start_index][0], offsets[end_index][1]
                    valid_answers.append(
                        {
                            "score": start_logit[start_index] + end_logit[end_index],
                            "text": context[start_char:end_char],
                        }
                    )

        if valid_answers:
            best = max(valid_answers, key=lambda x: x["score"])
            predictions[example["id"]] = best["text"]
        else:
            predictions[example["id"]] = ""
    return predictions


print("Decoding answer spans...")
predictions = postprocess_qa_predictions(
    raw_datasets["validation"], validation_dataset, start_logits, end_logits
)

with open(PREDICTIONS_FILE, "w") as f:
    json.dump(predictions, f)
print(f"Predictions written to {PREDICTIONS_FILE}")

Decoding answer spans...
Predictions written to predictions.json


In [14]:
print("Scoring with official evaluate-v2.0.py...")
result = subprocess.run(
    ["python", EVAL_SCRIPT, DEV_FILE, PREDICTIONS_FILE],
    check=True, capture_output=True, text=True,
)
print(result.stdout)

Scoring with official evaluate-v2.0.py...
{
  "exact": 80.91769157994324,
  "f1": 88.40156269460289,
  "total": 10570,
  "HasAns_exact": 80.91769157994324,
  "HasAns_f1": 88.40156269460289,
  "HasAns_total": 10570
}

